# Clase 6 — ¿Conviene ir a pescar a esta zona?
## 🟢 Nivel NOVATO — Machine Learning sin programar

**Curso:** IA Aplicada a la Producción Pesquera · UTN FRCh · PesquerosEnIA · 2026
**Autor:** Ariel Giamportone

---

**Para vos que nunca programaste.** No hay que escribir código: apretá ▶ y mirá el resultado.

**La idea:** una computadora puede aprender de **mareas pasadas** para estimar si una zona
va a ser buena o no, mirando la temperatura del agua, la clorofila y la profundidad.
A eso se le llama **Machine Learning** (aprendizaje automático).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)

print('✓ Librerías cargadas')

## 1. Los datos: 1.000 mareas del pasado

El bloque de abajo arma una tabla con 1.000 viajes de pesca. De cada uno sabemos las
condiciones del mar y si la marea **salió bien (1) o no (0)**. Apretá ▶.

In [ ]:
# ── Dataset: mareas históricas PCA (1.000 registros) ─────────────────────────
np.random.seed(42)
n_mareas = 1000

# Variables oceanográficas
temperatura_superficie = np.random.normal(10, 4, n_mareas)
clorofila_a = np.abs(np.random.exponential(1.8, n_mareas))
profundidad_media = np.random.uniform(50, 300, n_mareas)
salinidad = np.random.normal(33.5, 1.2, n_mareas)
velocidad_corriente = np.abs(np.random.normal(0.3, 0.15, n_mareas))

# Variables espacio-temporales
mes = np.random.randint(1, 13, n_mareas)
latitud = np.random.uniform(-52, -38, n_mareas)
longitud = np.random.uniform(-62, -48, n_mareas)

# Variable objetivo: captura_exitosa
# La captura es mayor cuando: SST ~8-12°C, clorofila alta, profundidad ~80-200m
prob_exito = (
    np.exp(-((temperatura_superficie - 10) ** 2) / 32) * 0.40 +
    np.clip(clorofila_a / 5, 0, 1) * 0.30 +
    np.exp(-((profundidad_media - 140) ** 2) / 5000) * 0.30
)
captura_exitosa = (prob_exito + np.random.normal(0, 0.15, n_mareas) > 0.40).astype(int)

datos_mareas = pd.DataFrame({
    'temperatura_superficie': temperatura_superficie,
    'clorofila_a': clorofila_a,
    'profundidad_media': profundidad_media,
    'salinidad': salinidad,
    'velocidad_corriente': velocidad_corriente,
    'mes': mes,
    'latitud': latitud,
    'longitud': longitud,
    'captura_exitosa': captura_exitosa
})

print(f'Dataset: {datos_mareas.shape[0]} mareas × {datos_mareas.shape[1]} variables')
print(f'Tasa de éxito: {datos_mareas["captura_exitosa"].mean():.1%}')
datos_mareas.head()

## 2. ¿Qué diferencia a una buena marea?

Miremos el promedio de condiciones en las mareas buenas vs. las que no. Apretá ▶.

In [ ]:
# ── Estadística descriptiva por clase ─────────────────────────────────────────
print('Estadísticas por resultado de marea:')
print(datos_mareas.groupby('captura_exitosa')[[
    'temperatura_superficie', 'clorofila_a', 'profundidad_media'
]].mean().round(2).rename(index={0: 'No exitosa', 1: 'Exitosa'}))

### 👀 Se nota un patrón

Las mareas exitosas suelen tener el agua **templada** y **más clorofila** (más comida).
La computadora va a aprender exactamente ese patrón.

## 3. La computadora aprende (y le tomamos examen)

Preparamos los datos y dejamos que el modelo **aprenda con una parte** y lo **evaluamos
con otra que nunca vio** (como un examen). Apretá ▶ en los dos bloques.

In [ ]:
# ── Split entrenamiento / prueba ──────────────────────────────────────────────
features = [
    'temperatura_superficie', 'clorofila_a', 'profundidad_media',
    'salinidad', 'velocidad_corriente', 'mes', 'latitud', 'longitud'
]
target = 'captura_exitosa'

X = datos_mareas[features]
y = datos_mareas[target]

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar features (importante para Regresión Logística)
escalador = StandardScaler()
X_entrenamiento_s = escalador.fit_transform(X_entrenamiento)
X_prueba_s = escalador.transform(X_prueba)

print(f'Entrenamiento: {X_entrenamiento.shape[0]} mareas')
print(f'Prueba:        {X_prueba.shape[0]} mareas')
print(f'Tasa de éxito entrenamiento: {y_entrenamiento.mean():.1%}')
print(f'Tasa de éxito prueba:        {y_prueba.mean():.1%}')

In [ ]:
# La computadora aprende de las mareas pasadas
from sklearn.ensemble import RandomForestClassifier
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_entrenamiento_s, y_entrenamiento)

# Le tomamos examen con mareas que nunca vio
precision = modelo_rf.score(X_prueba_s, y_prueba)
print(f'👉 El modelo acierta el {precision:.0%} de las veces si una zona será buena o no.')
print('   (aprendió mirando las mareas del pasado, no de memoria)')


## 4. La decisión: ¿a qué zona ir mañana?

Le damos al modelo las condiciones de **tres zonas posibles** para mañana y nos dice
la probabilidad de éxito de cada una. Apretá ▶.

In [ ]:
# ── Tres zonas candidatas para mañana (julio) ─────────────────────────────────
zonas_candidatas = pd.DataFrame({
    'temperatura_superficie': [9.2,  14.5,  6.8],
    'clorofila_a':            [3.1,   0.9,  1.2],
    'profundidad_media':      [130,   75,   210],
    'salinidad':              [33.4, 34.1, 33.0],
    'velocidad_corriente':    [0.28,  0.20,  0.45],
    'mes':                    [7,     7,     7],
    'latitud':                [-43.5, -40.8, -46.2],
    'longitud':               [-60.2, -58.5, -59.8]
}, index=[
    'Zona A — Frente a Rawson (~43°S)',
    'Zona B — Norte (40°S, aguas más cálidas)',
    'Zona C — Sur (46°S, zona profunda)'
])

# Predicción
zonas_scaled = escalador.transform(zonas_candidatas)
probabilidades = modelo_rf.predict_proba(zonas_scaled)[:, 1]

resultados_zonas = zonas_candidatas[[
    'temperatura_superficie', 'clorofila_a', 'profundidad_media'
]].copy()
resultados_zonas.columns = ['SST (°C)', 'Clorofila', 'Profundidad (m)']
resultados_zonas['Prob. éxito'] = [f'{p:.0%}' for p in probabilidades]
resultados_zonas['Recomendación'] = ['✅ IR' if p == max(probabilidades) else '—'
                                      for p in probabilidades]

print('Recomendación del modelo para el próximo viaje (julio):')
print(resultados_zonas.to_string())
print()
zona_recomendada = zonas_candidatas.index[probabilidades.argmax()]
print(f'→ Zona recomendada: {zona_recomendada}')
print(f'  Probabilidad de captura exitosa: {max(probabilidades):.0%}')
print()
print('Fundamento biológico:')
print(f'  SST {zonas_candidatas["temperatura_superficie"].iloc[probabilidades.argmax()]:.1f}°C'
      f' — dentro del rango óptimo para merluza (8-12°C)')

### 👀 Para tu barco

El modelo **recomienda la zona** con mayor probabilidad de éxito. No reemplaza al capitán:
lo **ayuda** a decidir con datos, ahorrando combustible y tiempo.

## ✅ Qué te llevás
- Una computadora puede **aprender de mareas pasadas** para estimar zonas buenas.
- Usa datos que ya existen: temperatura, clorofila, profundidad.
- Te da una **probabilidad**, no una certeza: es una ayuda para decidir mejor.

Cuando quieras, pasá al **Nivel Intermedio** para ver y tocar el código.
Comunidad: github.com/PesquerosEnIA